# Hate Speech Project — Mistral 7B only

This notebook preserves the original resume, cache, checkpoint, baseline evaluation, completion-only fine-tuning, and post-training evaluation logic. It runs only `unsloth/mistral-7b-instruct-v0.3-bnb-4bit`.


# Hate Speech Detection with Two Causal Language Models

This Colab notebook compares two instruction-tuned causal language models before and after QLoRA fine-tuning on HateXplain.

The experiment uses:

- the official HateXplain train, validation and test divisions;
- the original three labels: **Hate**, **Normal** and **Offensive**;
- one shared classification prompt;
- the tokenizer-specific chat template of each model;
- one strict output parser;
- the same official test set for every model and stage;
- one model in GPU memory at a time.

The test set is used only for final baseline and fine-tuned evaluation. It is not used for training or checkpoint selection.

## 0. Full static audit — completed before code changes

The original notebook was reviewed cell by cell before modification. The following issues were found.

### Critical execution blockers

1. **HateXplain labels were parsed with `int()`** although the official source JSON stores annotator labels as strings (`hatespeech`, `normal`, `offensive`). This stops execution during dataset construction.
2. **All durable outputs were written under `/content`**, so a Colab disconnect erased predictions, checkpoints, metrics and reports.

### Resume/cache gaps

3. Raw HateXplain JSON files were downloaded on every run.
4. Reconstructed official splits were rebuilt on every run and never saved with `Dataset.save_to_disk()`.
5. Conversational SFT preprocessing was repeated on every run.
6. Prediction CSV files were overwritten; completed predictions were not loaded, and interrupted inference could not continue from completed IDs.
7. Existing trainer checkpoints were ignored because `trainer.train()` was always called without `resume_from_checkpoint`.
8. An existing saved adapter did not skip training.
9. Metrics, confusion matrices, error analyses and comparison reports were regenerated unconditionally.
10. Writes were not atomic, so a runtime interruption could leave a partially written CSV or JSON file that looked reusable.

### Duplicate or avoidable work

11. Baseline inference ran again even when a complete compatible prediction file existed.
12. Fine-tuned inference ran again even when a complete compatible prediction file existed.
13. `matplotlib.pyplot` was imported twice.
14. GPU cleanup code was duplicated, while the helper that attempted `del obj` only deleted local references and therefore did not reliably release caller-owned objects.
15. The base model remained the object used for LoRA attachment even when a completed adapter already existed; there was no load-and-skip path.

### Robustness and reproducibility gaps

16. Cached artifacts had no experiment signature, so stale predictions could be mixed with a changed prompt, parser, label mapping, sequence length or test set.
17. There was no validation that a cached prediction file contained every official test ID exactly once.
18. There was no manifest recording the configuration and installed package versions.
19. The final artifact check required all outputs but had no cache-aware recovery path for missing derived reports.
20. The installation cell used unconstrained upgrades. This remains intentionally minimally changed for Colab compatibility, but exact installed versions are now recorded in the run manifest.

### Architecture checks

The required architecture was already correct and is preserved: two instruct causal LMs, QLoRA/LoRA through Unsloth and TRL, official train/validation/test divisions, one shared prompt, one strict parser, and evaluation through `generate()` rather than sequence-classification logits.

### SFTTrainer compatibility correction (v1.2)

The cached SFT dataset remains in prompt/completion conversational format. A tokenizer-aware `formatting_func` is now passed to `SFTTrainer`, supporting both single-example and batched calls made by Unsloth 2025.6.12. The training signature includes an SFT format version so incompatible prior checkpoints are not silently reused.


## 1. Install the required libraries

The notebook uses the current public interfaces of Unsloth, TRL and Transformers. The installation cell upgrades the related packages together to reduce version conflicts.

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)

!pip show torch torchao transformers trl peft bitsandbytes unsloth unsloth-zoo

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA: 12.8
Name: torch
Version: 2.11.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvshmem-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchdata, torchvision
---
Name: torchao
Version: 0.10.0
Summary: Package for applying ao techniques to GPU models
Home-page: https://github.com/pytorch/ao
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 
---
Name: transformers
Version: 5.15.0
Summary: Transformers: the model-definition framework fo

In [ ]:
# Step 2 — Mistral / current Colab environment
# Keep Colab's Torch 2.11.0 + CUDA 12.8 unchanged.

%pip install -q --no-cache-dir \
    "unsloth" \
    "transformers<=5.5.0,>=4.51.3" \
    "trl<=0.24.0,>=0.18.2" \
    "peft>=0.18.0" \
    "datasets>=3.4.1,<4.4.0" \
    "accelerate>=0.34.1" \
    "bitsandbytes>=0.45.5"

%pip uninstall -q -y hf-xet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 198.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 132.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 203.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 242.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 242.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 157.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 178.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 194.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 250.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 189.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 190.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 193.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

print("Hugging Face login succeeded.")

Hugging Face login succeeded.


In [ ]:
import shutil
from pathlib import Path

hub_cache = Path.home() / ".cache" / "huggingface" / "hub"

for folder in hub_cache.glob(
    "models--unsloth--llama-3.2-3b-instruct-unsloth-bnb-4bit*"
):
    print("Removing:", folder)
    shutil.rmtree(folder, ignore_errors=True)

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
CUDA version: 12.8
GPU: Tesla T4


## 2. Imports and reproducibility

In [ ]:
from unsloth import FastLanguageModel
import gc
import hashlib
import json
import os
import random
import re
import shutil
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import transformers
import datasets
import trl
import peft
import accelerate
import bitsandbytes
import matplotlib.pyplot as plt

from datasets import Dataset, DatasetDict, load_from_disk
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback, TrainerCallback
from trl import SFTConfig, SFTTrainer
#from unsloth import FastLanguageModel

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU runtime is required. In Colab, select Runtime > Change runtime type > GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4
PyTorch: 2.11.0+cu128
Transformers: 5.5.0
Datasets: 4.3.0
TRL: 0.24.0
PEFT: 0.20.0
Accelerate: 1.14.0
BitsAndBytes: 0.50.1


## 3. Central configuration

All important experiment settings are kept in one place. The two model identifiers can be changed here without changing the rest of the notebook.

In [ ]:
DATASET_NAME = "HateXplain"
MODEL_NAMES = ["unsloth/mistral-7b-instruct-v0.3-bnb-4bit"]

LABEL_ID_TO_NAME = {0: "Hate", 1: "Normal", 2: "Offensive"}
LABEL_NAME_TO_ID = {name: idx for idx, name in LABEL_ID_TO_NAME.items()}
HATEXPLAIN_SOURCE_LABEL_TO_ID = {
    "hatespeech": 0,
    "hate": 0,
    "normal": 1,
    "offensive": 2,
}
VALID_LABELS = tuple(LABEL_NAME_TO_ID.keys())

MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 6
INFERENCE_BATCH_SIZE = 8

LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 20
EVAL_STEPS = 100
SAVE_STEPS = 100
SAVE_TOTAL_LIMIT = 1
EARLY_STOPPING_PATIENCE = 2

# Problem: /content is ephemeral. Cause: Colab deletes it when the runtime disconnects.
# Solution: mount Drive and keep every resumable artifact in one persistent project directory.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/Hate_Speech_Final_Project")
except ImportError:
    PROJECT_ROOT = Path.cwd() / "Hate_Speech_Final_Project"

RESULTS_DIR = PROJECT_ROOT / "results"
CACHE_DIR = PROJECT_ROOT / "cache"
DATASET_CACHE_DIR = CACHE_DIR / "official_hatexplain_dataset"
SFT_CACHE_DIR = CACHE_DIR / "sft_datasets"
RAW_CACHE_DIR = CACHE_DIR / "raw"

for directory in (RESULTS_DIR, CACHE_DIR, RAW_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Persistent project directory:", PROJECT_ROOT)
print("Results directory:", RESULTS_DIR)


Mounted at /content/drive
Persistent project directory: /content/drive/MyDrive/Hate_Speech_Final_Project
Results directory: /content/drive/MyDrive/Hate_Speech_Final_Project/results


## 4. Load or reconstruct the official HateXplain divisions

**Problem:** the original loader converted string labels with `int()` and rebuilt/downloaded the dataset on every run.  
**Cause:** the upstream JSON uses textual annotator labels, while no persistent raw or processed cache existed.  
**Solution:** normalize the official strings explicitly, cache the two source JSON files, save the reconstructed `DatasetDict` to Drive, and load it on later runs. The official split files and label IDs remain unchanged.


In [ ]:
HATEXPLAIN_DATA_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json"
HATEXPLAIN_SPLITS_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/post_id_divisions.json"


def atomic_write_json(payload, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


def atomic_write_csv(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def download_json_cached(url, cache_path):
    cache_path = Path(cache_path)
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    payload = response.json()
    atomic_write_json(payload, cache_path)
    return payload


def normalize_hatexplain_label(value):
    if isinstance(value, (int, np.integer)):
        label_id = int(value)
    else:
        key = str(value).strip().lower().replace("_", "").replace(" ", "")
        if key.isdigit():
            label_id = int(key)
        elif key in HATEXPLAIN_SOURCE_LABEL_TO_ID:
            label_id = HATEXPLAIN_SOURCE_LABEL_TO_ID[key]
        else:
            raise ValueError(f"Unexpected HateXplain annotator label: {value!r}")
    if label_id not in LABEL_ID_TO_NAME:
        raise ValueError(f"Unexpected HateXplain label ID: {label_id}")
    return label_id


def majority_label(annotators):
    labels = [normalize_hatexplain_label(item["label"]) for item in annotators]
    label_id, count = Counter(labels).most_common(1)[0]
    return label_id if count >= 2 else None


def build_split(raw_posts, post_ids, split_name):
    rows, unresolved = [], []
    for post_id in post_ids:
        item = raw_posts[str(post_id)]
        label_id = majority_label(item["annotators"])
        if label_id is None:
            unresolved.append(str(post_id))
            continue
        text = " ".join(str(token) for token in item["post_tokens"]).strip()
        rows.append({
            "id": str(post_id),
            "text": text,
            "label": int(label_id),
            "label_name": LABEL_ID_TO_NAME[int(label_id)],
            "split": split_name,
        })
    return Dataset.from_list(rows), unresolved


if DATASET_CACHE_DIR.exists():
    dataset = load_from_disk(str(DATASET_CACHE_DIR))
    unresolved_path = CACHE_DIR / "unresolved_annotation_ties.csv"
    unresolved_df = pd.read_csv(unresolved_path, dtype={"id": str}) if unresolved_path.exists() else pd.DataFrame(columns=["id", "split"])
    print("Loaded cached official HateXplain dataset.")
else:
    raw_posts = download_json_cached(HATEXPLAIN_DATA_URL, RAW_CACHE_DIR / "dataset.json")
    official_divisions = download_json_cached(HATEXPLAIN_SPLITS_URL, RAW_CACHE_DIR / "post_id_divisions.json")

    train_ds, unresolved_train = build_split(raw_posts, official_divisions["train"], "train")
    validation_ds, unresolved_validation = build_split(raw_posts, official_divisions["val"], "validation")
    test_ds, unresolved_test = build_split(raw_posts, official_divisions["test"], "test")
    dataset = DatasetDict({"train": train_ds, "validation": validation_ds, "test": test_ds})

    unresolved_df = pd.DataFrame([
        *({"id": x, "split": "train"} for x in unresolved_train),
        *({"id": x, "split": "validation"} for x in unresolved_validation),
        *({"id": x, "split": "test"} for x in unresolved_test),
    ], columns=["id", "split"])

    tmp_dataset_dir = DATASET_CACHE_DIR.with_name(DATASET_CACHE_DIR.name + ".tmp")
    if tmp_dataset_dir.exists():
        shutil.rmtree(tmp_dataset_dir)
    dataset.save_to_disk(str(tmp_dataset_dir))
    tmp_dataset_dir.replace(DATASET_CACHE_DIR)
    atomic_write_csv(unresolved_df, CACHE_DIR / "unresolved_annotation_ties.csv")
    print("Reconstructed and cached official HateXplain dataset.")

train_ds = dataset["train"]
validation_ds = dataset["validation"]
test_ds = dataset["test"]
atomic_write_csv(unresolved_df, RESULTS_DIR / "unresolved_annotation_ties.csv")

print(dataset)
print("\nLabel mapping:", LABEL_ID_TO_NAME)
print("Excluded posts without a strict majority:", len(unresolved_df))


Loaded cached official HateXplain dataset.
DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_name', 'split'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_name', 'split'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_name', 'split'],
        num_rows: 1924
    })
})

Label mapping: {0: 'Hate', 1: 'Normal', 2: 'Offensive'}
Excluded posts without a strict majority: 0


## 5. Inspect class distributions

In [ ]:
def class_distribution(ds):
    counts = pd.Series(ds["label_name"]).value_counts()
    return counts.reindex(VALID_LABELS, fill_value=0)

distribution_df = pd.DataFrame(
    {
        "train": class_distribution(train_ds),
        "validation": class_distribution(validation_ds),
        "test": class_distribution(test_ds),
    }
)
display(distribution_df)

assert set(train_ds.unique("label")) <= {0, 1, 2}
assert set(validation_ds.unique("label")) <= {0, 1, 2}
assert set(test_ds.unique("label")) <= {0, 1, 2}
assert set(train_ds["id"]).isdisjoint(validation_ds["id"])
assert set(train_ds["id"]).isdisjoint(test_ds["id"])
assert set(validation_ds["id"]).isdisjoint(test_ds["id"])

print("Official split separation checks passed.")

,train,validation,test
Hate,4748,593,594
Normal,6251,781,782
Offensive,4384,548,548


Official split separation checks passed.


## 6. Shared prompt and strict parser

Baseline and fine-tuned inference use the same system message, user message, chat template, generation settings and parser.

The parser accepts only one complete label: `Hate`, `Normal` or `Offensive`. Any additional text, punctuation or explanation is invalid and is stored separately.

In [ ]:
SYSTEM_MESSAGE = (
    "You are a text classification model. "
    "Classify the given text into exactly one of these labels: "
    "Hate, Normal, Offensive. "
    "Return only the label and nothing else."
)

def build_messages(text):
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {
            "role": "user",
            "content": (
                "Classify the following text.\n\n"
                f"Text: {text}\n\n"
                "Valid labels: Hate, Normal, Offensive."
            ),
        },
    ]

_STRICT_LABEL_PATTERN = re.compile(r"^(Hate|Normal|Offensive)$")

def parse_label(raw_output):
    cleaned = raw_output.strip()
    match = _STRICT_LABEL_PATTERN.fullmatch(cleaned)
    if match is None:
        return None
    return match.group(1)

for expected in VALID_LABELS:
    assert parse_label(expected) == expected
assert parse_label("Hate.") is None
assert parse_label("The answer is Hate") is None
assert parse_label("hate") is None

print("Strict parser checks passed.")

def stable_hash(payload):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:16]

PROMPT_PARSER_SIGNATURE = stable_hash({
    "system_message": SYSTEM_MESSAGE,
    "labels": LABEL_ID_TO_NAME,
    "parser_pattern": _STRICT_LABEL_PATTERN.pattern,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "test_ids": list(test_ds["id"]),
})
print("Experiment signature:", PROMPT_PARSER_SIGNATURE)


Strict parser checks passed.
Experiment signature: 2e1486a183518def


## 7. Shared generation pipeline with resumable predictions

**Problem:** prediction files were overwritten and interrupted inference restarted from the first test example.  
**Cause:** generation had no artifact signature, completeness validation or partial-ID resume.  
**Solution:** use one experiment signature, load compatible completed rows, generate only missing official test IDs, and atomically update prediction and invalid-output files after every batch.


In [ ]:
def ensure_tokenizer_settings(tokenizer):
    if tokenizer.chat_template is None:
        raise ValueError("Tokenizer has no chat template; an instruction-tuned checkpoint is required.")
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    return tokenizer


def expected_prediction_signature(stage):
    if stage == "baseline":
        return PROMPT_PARSER_SIGNATURE
    return stable_hash({
        "prompt_parser_signature": PROMPT_PARSER_SIGNATURE,
        "training_signature": TRAINING_SIGNATURE,
    })


def prediction_paths(model_name, stage):
    slug = model_name.replace("/", "__")
    return (
        RESULTS_DIR / f"{slug}__{stage}__predictions.csv",
        RESULTS_DIR / f"{slug}__{stage}__invalid_outputs.csv",
    )


def load_compatible_predictions(path, ds, model_name, stage):
    expected_columns = {
        "id", "text", "true_label_id", "true_label", "raw_output", "parsed_label",
        "prediction_id", "is_valid", "model", "stage", "signature",
    }
    if not path.exists():
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, dtype={"id": str})
    except Exception:
        return pd.DataFrame()
    if not expected_columns.issubset(df.columns) or df.empty:
        return pd.DataFrame()
    compatible = (
        df["model"].eq(model_name)
        & df["stage"].eq(stage)
        & df["signature"].eq(expected_prediction_signature(stage))
        & df["id"].isin(set(ds["id"]))
    )
    df = df.loc[compatible].drop_duplicates("id", keep="last").copy()
    return df


@torch.inference_mode()
def generate_predictions(model, tokenizer, ds, model_name, stage):
    predictions_path, invalid_path = prediction_paths(model_name, stage)
    predictions_df = load_compatible_predictions(predictions_path, ds, model_name, stage)
    completed_ids = set(predictions_df["id"]) if not predictions_df.empty else set()
    missing_indices = [i for i, item_id in enumerate(ds["id"]) if item_id not in completed_ids]

    if not missing_indices:
        print(f"{model_name} | {stage}: loaded {len(predictions_df)} cached predictions; inference skipped.")
        return predictions_df.set_index("id").loc[list(ds["id"])].reset_index()

    FastLanguageModel.for_inference(model)
    for offset in tqdm(
    range(0, len(missing_indices), INFERENCE_BATCH_SIZE),
    total=(len(missing_indices) + INFERENCE_BATCH_SIZE - 1) // INFERENCE_BATCH_SIZE,
    desc=f"{model_name} | {stage}",
):
        indices = missing_indices[offset:offset + INFERENCE_BATCH_SIZE]
        batch = ds.select(indices)
        prompt_texts = [
            tokenizer.apply_chat_template(build_messages(text), tokenize=False, add_generation_prompt=True)
            for text in batch["text"]
        ]
        encoded = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        )
        encoded = {key: value.to(model.device) for key, value in encoded.items()}
        generated = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        input_width = encoded["input_ids"].shape[1]
        records = []
        for i in range(len(batch)):
            raw_output = tokenizer.decode(generated[i, input_width:], skip_special_tokens=True).strip()
            parsed_label = parse_label(raw_output)
            records.append({
                "id": batch["id"][i],
                "text": batch["text"][i],
                "true_label_id": int(batch["label"][i]),
                "true_label": batch["label_name"][i],
                "raw_output": raw_output,
                "parsed_label": parsed_label,
                "prediction_id": LABEL_NAME_TO_ID[parsed_label] if parsed_label is not None else -1,
                "is_valid": parsed_label is not None,
                "model": model_name,
                "stage": stage,
                "signature": expected_prediction_signature(stage),
            })
        predictions_df = pd.concat([predictions_df, pd.DataFrame(records)], ignore_index=True)
        predictions_df = predictions_df.drop_duplicates("id", keep="last")
        ordered = predictions_df.set_index("id").reindex([x for x in ds["id"] if x in set(predictions_df["id"])]).reset_index()
        atomic_write_csv(ordered, predictions_path)
        atomic_write_csv(ordered.loc[~ordered["is_valid"].astype(bool)], invalid_path)
        del encoded, generated
        torch.cuda.empty_cache()

    predictions_df = pd.read_csv(predictions_path, dtype={"id": str})
    expected_ids = list(ds["id"])
    if predictions_df["id"].tolist() != expected_ids or len(predictions_df) != len(expected_ids):
        raise RuntimeError(f"Prediction cache is incomplete or out of order: {predictions_path}")
    print(f"{model_name} | {stage}: {len(predictions_df)} predictions, {(predictions_df['prediction_id'] == -1).sum()} invalid outputs")
    return predictions_df


## 8. Cached metrics, confusion matrices and error analysis

**Problem:** all derived evaluation artifacts were recomputed on every run.  
**Cause:** evaluation functions did not check for compatible existing outputs.  
**Solution:** cache metrics by the same experiment signature and independently skip existing confusion matrices and error files; missing artifacts are regenerated from saved predictions without rerunning inference.


In [ ]:
def metric_artifact_paths(predictions_df):
    slug = predictions_df["model"].iloc[0].replace("/", "__")
    stage = predictions_df["stage"].iloc[0]
    return (
        RESULTS_DIR / f"{slug}__{stage}__metrics.json",
        RESULTS_DIR / f"{slug}__{stage}__confusion_matrix.png",
        RESULTS_DIR / f"{slug}__{stage}__errors.csv",
    )


def compute_metrics_from_predictions(predictions_df):
    y_true = predictions_df["true_label_id"].astype(int).to_numpy()
    y_pred = predictions_df["prediction_id"].astype(int).to_numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average="macro", zero_division=0
    )
    return {
        "model": predictions_df["model"].iloc[0],
        "stage": predictions_df["stage"].iloc[0],
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "invalid_count": int((y_pred == -1).sum()),
        "invalid_rate": float((y_pred == -1).mean()),
        "test_examples": int(len(predictions_df)),
        "signature": expected_prediction_signature(predictions_df["stage"].iloc[0]),
    }


def save_confusion_matrix(predictions_df, path):
    matrix = np.zeros((3, 4), dtype=int)
    for true_id, pred_id in zip(predictions_df["true_label_id"].astype(int), predictions_df["prediction_id"].astype(int)):
        matrix[true_id, pred_id if pred_id in {0, 1, 2} else 3] += 1
    fig, ax = plt.subplots(figsize=(8, 5))
    image = ax.imshow(matrix)
    ax.set_xticks(range(4), labels=["Hate", "Normal", "Offensive", "Invalid"])
    ax.set_yticks(range(3), labels=["Hate", "Normal", "Offensive"])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"{predictions_df['model'].iloc[0]} — {predictions_df['stage'].iloc[0]}")
    for row in range(3):
        for col in range(4):
            ax.text(col, row, str(matrix[row, col]), ha="center", va="center")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    tmp = path.with_suffix(".tmp.png")
    fig.savefig(tmp, dpi=200, bbox_inches="tight")
    plt.close(fig)
    tmp.replace(path)


def evaluate_saved_predictions(predictions_df):
    metrics_path, matrix_path, errors_path = metric_artifact_paths(predictions_df)
    metrics = None
    if metrics_path.exists():
        candidate = json.loads(metrics_path.read_text(encoding="utf-8"))
        if candidate.get("signature") == expected_prediction_signature(predictions_df["stage"].iloc[0]):
            metrics = candidate
    if metrics is None:
        metrics = compute_metrics_from_predictions(predictions_df)
        atomic_write_json(metrics, metrics_path)
    # Regenerate these lightweight artifacts so they always match the compatible predictions.
    save_confusion_matrix(predictions_df, matrix_path)
    errors_df = predictions_df.loc[
        predictions_df["prediction_id"] != predictions_df["true_label_id"]
    ].copy()
    atomic_write_csv(errors_df, errors_path)

    print(classification_report(
        predictions_df["true_label_id"].astype(int),
        predictions_df["prediction_id"].astype(int),
        labels=[0, 1, 2],
        target_names=["Hate", "Normal", "Offensive"],
        zero_division=0,
    ))
    return metrics


## 9. Load or build the conversational fine-tuning datasets

**Problem:** prompt/completion preprocessing was repeated after every Colab restart.  
**Cause:** mapped datasets existed only in RAM.  
**Solution:** save the processed train and validation datasets to the persistent cache and load them when present.


In [ ]:
def to_training_example(example):
    return {
        "prompt": build_messages(example["text"]),
        "completion": [{"role": "assistant", "content": example["label_name"]}],
    }

ACTIVE_SFT_CACHE_DIR = SFT_CACHE_DIR / PROMPT_PARSER_SIGNATURE

if ACTIVE_SFT_CACHE_DIR.exists():
    sft_dataset = load_from_disk(str(ACTIVE_SFT_CACHE_DIR))
    print("Loaded cached SFT preprocessing.")
else:
    sft_dataset = DatasetDict({
        "train": train_ds.map(to_training_example, remove_columns=train_ds.column_names),
        "validation": validation_ds.map(to_training_example, remove_columns=validation_ds.column_names),
    })
    tmp_sft_dir = ACTIVE_SFT_CACHE_DIR.with_name(ACTIVE_SFT_CACHE_DIR.name + ".tmp")
    if tmp_sft_dir.exists():
        shutil.rmtree(tmp_sft_dir)
    sft_dataset.save_to_disk(str(tmp_sft_dir))
    ACTIVE_SFT_CACHE_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp_sft_dir.replace(ACTIVE_SFT_CACHE_DIR)
    print("Built and cached SFT preprocessing.")

train_sft = sft_dataset["train"]
validation_sft = sft_dataset["validation"]
print(train_sft[0])


Loaded cached SFT preprocessing.
{'prompt': [{'content': 'You are a text classification model. Classify the given text into exactly one of these labels: Hate, Normal, Offensive. Return only the label and nothing else.', 'role': 'system'}, {'content': 'Classify the following text.\n\nText: u really think i would not have been raped by feral hindu or muslim back in india or bangladesh and a neo nazi would rape me as well just to see me cry\n\nValid labels: Hate, Normal, Offensive.', 'role': 'user'}], 'completion': [{'content': 'Offensive', 'role': 'assistant'}]}


## 10. Load, evaluate, resume fine-tuning and release one model at a time

**Problem:** training always restarted, completed adapters were ignored, and GPU cleanup was duplicated.  
**Cause:** no checkpoint discovery, no adapter-completion marker, and no single cleanup routine.  
**Solution:** skip baseline inference when cached, resume the latest trainer checkpoint after interruption, skip training when a validated adapter marker exists, reload that adapter for final inference, and explicitly remove all model/trainer references before the next model.


In [ ]:
from pathlib import Path

cache_dirs = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path("/root/.cache/huggingface/hub"),
]

for cache_dir in cache_dirs:
    print(cache_dir, "exists:", cache_dir.exists())

/root/.cache/huggingface/hub exists: True
/root/.cache/huggingface/hub exists: True


In [ ]:
def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def latest_checkpoint(checkpoint_dir):
    checkpoints = []
    for path in Path(checkpoint_dir).glob("checkpoint-*"):
        try:
            checkpoints.append((int(path.name.rsplit("-", 1)[1]), path))
        except ValueError:
            pass
    return max(checkpoints, default=(None, None))[1]


def adapter_is_complete(adapter_dir, marker_path, model_name):
    if not adapter_dir.exists() or not marker_path.exists():
        return False
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return marker.get("model") == model_name and marker.get("training_signature") == TRAINING_SIGNATURE


TRAINING_SIGNATURE = stable_hash({
    "prompt_parser_signature": PROMPT_PARSER_SIGNATURE,
    "train_ids": list(train_ds["id"]),
    "validation_ids": list(validation_ds["id"]),
    "lora": [LORA_R, LORA_ALPHA, LORA_DROPOUT],
    "training": [TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE, GRADIENT_ACCUMULATION_STEPS, LEARNING_RATE, NUM_TRAIN_EPOCHS],
    "sft_format_version": "pretokenized_completion_only_token_subsequence_v3",
})


def _as_token_id_list(value):
    if isinstance(value, torch.Tensor):
        value = value.tolist()
    if value and isinstance(value[0], list):
        if len(value) != 1:
            raise ValueError("Expected one tokenized sequence per training example.")
        value = value[0]
    return list(value)


def _find_subsequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return -1
    last_start = len(sequence) - len(subsequence)
    for start in range(last_start, -1, -1):
        if sequence[start:start + len(subsequence)] == subsequence:
            return start
    return -1


def tokenize_completion_only_example(example, tokenizer):
    prompt_messages = example["prompt"]
    completion_messages = example["completion"]

    if len(completion_messages) != 1 or completion_messages[0].get("role") != "assistant":
        raise ValueError("Expected exactly one assistant completion message.")

    completion_text = completion_messages[0]["content"]

    full_ids = _as_token_id_list(tokenizer.apply_chat_template(
        [*prompt_messages, *completion_messages],
        tokenize=True,
        add_generation_prompt=False,
    ))

    # Tokenize several boundary variants because Mistral may merge leading
    # whitespace with the first completion token.
    completion_variants = []
    for text_variant in (
        completion_text,
        " " + completion_text,
        "\n" + completion_text,
    ):
        token_ids = _as_token_id_list(tokenizer(
            text_variant,
            add_special_tokens=False,
        )["input_ids"])
        if token_ids and token_ids not in completion_variants:
            completion_variants.append(token_ids)

    completion_start = -1
    matched_completion_ids = None
    for completion_ids in completion_variants:
        start = _find_subsequence(full_ids, completion_ids)
        if start > completion_start:
            completion_start = start
            matched_completion_ids = completion_ids

    if completion_start < 0 or matched_completion_ids is None:
        raise RuntimeError(
            "Could not locate the assistant completion token sequence "
            "inside the rendered Mistral chat sequence."
        )

    input_ids = full_ids[:MAX_SEQ_LENGTH]
    completion_end = completion_start + len(matched_completion_ids)

    labels = []
    for index, token_id in enumerate(input_ids):
        if completion_start <= index < completion_end:
            labels.append(token_id)
        else:
            labels.append(-100)

    if not any(label != -100 for label in labels):
        raise RuntimeError(
            "MAX_SEQ_LENGTH truncates the assistant completion. "
            "Increase MAX_SEQ_LENGTH before training."
        )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


def get_completion_only_datasets(tokenizer, model_name):
    model_slug = model_name.replace("/", "__")
    cache_signature = stable_hash({
        "training_signature": TRAINING_SIGNATURE,
        "model": model_name,
        "max_seq_length": MAX_SEQ_LENGTH,
        "tokenizer_chat_template": tokenizer.chat_template,
    })
    cache_dir = SFT_CACHE_DIR / "completion_only_tokenized" / model_slug / cache_signature

    if cache_dir.exists():
        tokenized = load_from_disk(str(cache_dir))
        print(f"{model_name}: loaded cached completion-only tokenization.")
    else:
        tokenized = DatasetDict({
            "train": train_sft.map(
                lambda example: tokenize_completion_only_example(example, tokenizer),
                remove_columns=train_sft.column_names,
                desc=f"Tokenizing {model_name} train split",
            ),
            "validation": validation_sft.map(
                lambda example: tokenize_completion_only_example(example, tokenizer),
                remove_columns=validation_sft.column_names,
                desc=f"Tokenizing {model_name} validation split",
            ),
        })
        tmp_dir = cache_dir.with_name(cache_dir.name + ".tmp")
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)
        cache_dir.parent.mkdir(parents=True, exist_ok=True)
        tokenized.save_to_disk(str(tmp_dir))
        tmp_dir.replace(cache_dir)
        print(f"{model_name}: built and cached completion-only tokenization.")

    for split_name in ("train", "validation"):
        supervised_counts = [
            sum(label != -100 for label in labels)
            for labels in tokenized[split_name]["labels"]
        ]
        if not supervised_counts or min(supervised_counts) < 1:
            raise RuntimeError(f"{split_name} contains an example with no supervised completion tokens.")

    return tokenized["train"], tokenized["validation"]


def load_model(model_name_or_path):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(model_name_or_path),
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=LOAD_IN_4BIT,
    )
    tokenizer.padding_side= "left"
    return model, ensure_tokenizer_settings(tokenizer)


def run_model_experiment(model_name):
    model_slug = model_name.replace("/", "__")
    model_output_dir = RESULTS_DIR / model_slug
    checkpoint_dir = model_output_dir / "checkpoints" / TRAINING_SIGNATURE
    adapter_dir = model_output_dir / "best_adapter"
    adapter_marker = model_output_dir / "adapter_complete.json"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    model = tokenizer = trainer = None
    try:
        baseline_path, baseline_invalid_path = prediction_paths(model_name, "baseline")
        baseline_cached = load_compatible_predictions(baseline_path, test_ds, model_name, "baseline")
        baseline_complete = len(baseline_cached) == len(test_ds) and set(baseline_cached["id"]) == set(test_ds["id"])
        if baseline_complete:
            baseline_predictions = baseline_cached.set_index("id").loc[list(test_ds["id"])].reset_index()
            atomic_write_csv(baseline_predictions.loc[baseline_predictions["prediction_id"].astype(int) == -1], baseline_invalid_path)
            print(f"{model_name} | baseline: complete cache found; model generation skipped.")
        else:
            model, tokenizer = load_model(model_name)
            baseline_predictions = generate_predictions(model, tokenizer, test_ds, model_name, "baseline")
        baseline_metrics = evaluate_saved_predictions(baseline_predictions)

        fine_path, fine_invalid_path = prediction_paths(model_name, "fine_tuned")
        fine_cached = load_compatible_predictions(fine_path, test_ds, model_name, "fine_tuned")
        fine_complete = len(fine_cached) == len(test_ds) and set(fine_cached["id"]) == set(test_ds["id"])
        if fine_complete:
            fine_tuned_predictions = fine_cached.set_index("id").loc[list(test_ds["id"])].reset_index()
            atomic_write_csv(fine_tuned_predictions.loc[fine_tuned_predictions["prediction_id"].astype(int) == -1], fine_invalid_path)
            print(f"{model_name} | fine_tuned: complete cache found; adapter loading and inference skipped.")
            fine_tuned_metrics = evaluate_saved_predictions(fine_tuned_predictions)
            return [baseline_metrics, fine_tuned_metrics]

        if adapter_is_complete(adapter_dir, adapter_marker, model_name):
            print(f"{model_name}: completed adapter found; training skipped.")
            model = None
            tokenizer = None
            clear_gpu()
            model, tokenizer = load_model(adapter_dir)
        else:
            if model is None:
                model, tokenizer = load_model(model_name)
            model = FastLanguageModel.get_peft_model(
                model,
                r=LORA_R,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                lora_alpha=LORA_ALPHA,
                lora_dropout=LORA_DROPOUT,
                bias="none",
                use_gradient_checkpointing="unsloth",
                random_state=SEED,
                use_rslora=False,
                loftq_config=None,
            )
            training_args = SFTConfig(
                output_dir=str(checkpoint_dir),
                max_length=MAX_SEQ_LENGTH,
                per_device_train_batch_size=TRAIN_BATCH_SIZE,
                per_device_eval_batch_size=EVAL_BATCH_SIZE,
                gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
                learning_rate=LEARNING_RATE,
                num_train_epochs=NUM_TRAIN_EPOCHS,
                warmup_ratio=WARMUP_RATIO,
                weight_decay=WEIGHT_DECAY,
                logging_steps=LOGGING_STEPS,
                eval_strategy="steps",
                eval_steps=EVAL_STEPS,
                save_strategy="steps",
                save_steps=SAVE_STEPS,
                save_total_limit=SAVE_TOTAL_LIMIT,
                load_best_model_at_end=True,
                metric_for_best_model="eval_loss",
                greater_is_better=False,
                bf16=torch.cuda.is_bf16_supported(),
                fp16=not torch.cuda.is_bf16_supported(),
                optim="adamw_8bit",
                lr_scheduler_type="linear",
                report_to="none",
                seed=SEED,
                data_seed=SEED,
                packing=False,
            )

            tokenized_train_sft, tokenized_validation_sft = get_completion_only_datasets(
                tokenizer,
                model_name,
            )
            completion_only_collator = DataCollatorForSeq2Seq(
                tokenizer=tokenizer,
                padding=True,
                label_pad_token_id=-100,
                return_tensors="pt",
            )

            trainer = SFTTrainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_train_sft,
                eval_dataset=tokenized_validation_sft,
                processing_class=tokenizer,
                data_collator=completion_only_collator,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
            )
            resume_checkpoint = latest_checkpoint(checkpoint_dir)
            trainer.train(resume_from_checkpoint=str(resume_checkpoint) if resume_checkpoint else None)
            trainer.model.save_pretrained(adapter_dir)
            tokenizer.save_pretrained(adapter_dir)
            atomic_write_json({"model": model_name, "training_signature": TRAINING_SIGNATURE}, adapter_marker)
            model = trainer.model

        fine_tuned_predictions = generate_predictions(model, tokenizer, test_ds, model_name, "fine_tuned")
        fine_tuned_metrics = evaluate_saved_predictions(fine_tuned_predictions)
        return [baseline_metrics, fine_tuned_metrics]
    finally:
        trainer = None
        model = None
        tokenizer = None
        clear_gpu()


## 11. Run both model experiments sequentially and cache aggregate metrics

**Problem:** the aggregate metrics file was overwritten only after both models finished, losing progress after a disconnect.  
**Cause:** rows were accumulated only in RAM.  
**Solution:** merge and atomically save metrics after each model; existing compatible per-stage metrics are reused.


In [ ]:
metrics_path = RESULTS_DIR / "model_comparison_metrics.csv"
all_metric_rows = []

if metrics_path.exists():
    previous_metrics = pd.read_csv(metrics_path)
    if "signature" in previous_metrics.columns:
        allowed_signatures = {
            expected_prediction_signature("baseline"),
            expected_prediction_signature("fine_tuned"),
        }
        all_metric_rows.extend(
            previous_metrics.loc[
                previous_metrics["signature"].isin(allowed_signatures)
            ].to_dict("records")
        )

for model_name in MODEL_NAMES:
    print("=" * 100)
    print("Starting model:", model_name)
    print("=" * 100)
    new_rows = run_model_experiment(model_name)
    all_metric_rows.extend(new_rows)
    metrics_df = pd.DataFrame(all_metric_rows).drop_duplicates(["model", "stage"], keep="last")
    metrics_df = metrics_df.sort_values(["model", "stage"]).reset_index(drop=True)
    atomic_write_csv(metrics_df, metrics_path)
    clear_gpu()

metrics_df = pd.read_csv(metrics_path)
display(metrics_df)
print("Saved metrics to:", metrics_path)

run_manifest = {
    "prompt_parser_signature": PROMPT_PARSER_SIGNATURE,
    "training_signature": TRAINING_SIGNATURE,
    "models": MODEL_NAMES,
    "label_mapping": LABEL_ID_TO_NAME,
    "dataset_sizes": {split: len(dataset[split]) for split in dataset},
    "versions": {
        "torch": torch.__version__, "transformers": transformers.__version__,
        "datasets": datasets.__version__, "trl": trl.__version__,
        "peft": peft.__version__, "accelerate": accelerate.__version__,
        "bitsandbytes": bitsandbytes.__version__,
    },
}
atomic_write_json(run_manifest, RESULTS_DIR / "run_manifest.json")


Starting model: unsloth/mistral-7b-instruct-v0.3-bnb-4bit
unsloth/mistral-7b-instruct-v0.3-bnb-4bit | baseline: complete cache found; model generation skipped.
              precision    recall  f1-score   support

        Hate       0.34      0.05      0.08       594
      Normal       0.89      0.20      0.33       782
   Offensive       0.31      0.92      0.46       548

   micro avg       0.37      0.36      0.36      1924
   macro avg       0.51      0.39      0.29      1924
weighted avg       0.56      0.36      0.29      1924

unsloth/mistral-7b-instruct-v0.3-bnb-4bit | fine_tuned: complete cache found; adapter loading and inference skipped.
              precision    recall  f1-score   support

        Hate       0.00      0.00      0.00       594
      Normal       0.83      0.20      0.32       782
   Offensive       0.00      0.00      0.00       548

   micro avg       0.83      0.08      0.15      1924
   macro avg       0.28      0.07      0.11      1924
weighted avg    

,model,stage,accuracy,precision_macro,recall_macro,f1_macro,invalid_count,invalid_rate,test_examples,signature
0,unsloth/mistral-7b-instruct-v0.3-bnb-4bit,baseline,0.358628,0.514821,0.389631,0.292146,41,0.021310,1924,2e1486a183518def
1,unsloth/mistral-7b-instruct-v0.3-bnb-4bit,fine_tuned,0.081081,0.278075,0.066496,0.107327,1737,0.902807,1924,c2eb66ca6edbdf1a


Saved metrics to: /content/drive/MyDrive/Hate_Speech_Final_Project/results/model_comparison_metrics.csv


## 12. Load or generate the baseline vs. fine-tuned Macro F1 report

**Problem:** the comparison chart was recreated unconditionally.  
**Cause:** the report cell did not check for an existing compatible metrics table and image.  
**Solution:** use the cached aggregate metrics and create the chart only when it is missing.


In [ ]:
comparison_plot_path = RESULTS_DIR / "macro_f1_comparison.png"
if not comparison_plot_path.exists():
    plot_df = metrics_df.copy()
    plot_df["model_stage"] = plot_df["model"].str.split("/").str[-1] + " — " + plot_df["stage"]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_df["model_stage"], plot_df["f1_macro"])
    ax.set_ylabel("Macro F1")
    ax.set_title("Baseline and Fine-Tuned Model Comparison")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    tmp_plot = comparison_plot_path.with_suffix(".tmp.png")
    fig.savefig(tmp_plot, dpi=200, bbox_inches="tight")
    plt.close(fig)
    tmp_plot.replace(comparison_plot_path)
else:
    print("Loaded existing comparison report:", comparison_plot_path)


Loaded existing comparison report: /content/drive/MyDrive/Hate_Speech_Final_Project/results/macro_f1_comparison.png


## 13. Verify saved outputs and project invariants

The verification below checks durable artifacts, official split separation, prediction completeness, architecture constraints and one-model-at-a-time execution assumptions. It performs no inference or training.


In [ ]:
artifact_rows = [{"relative_path": str(path.relative_to(PROJECT_ROOT)), "size_bytes": path.stat().st_size}
                 for path in sorted(PROJECT_ROOT.rglob("*")) if path.is_file()]
artifacts_df = pd.DataFrame(artifact_rows)
display(artifacts_df)

required_suffixes = [
    "__baseline__predictions.csv", "__fine_tuned__predictions.csv",
    "__baseline__invalid_outputs.csv", "__fine_tuned__invalid_outputs.csv",
    "__baseline__errors.csv", "__fine_tuned__errors.csv",
    "__baseline__confusion_matrix.png", "__fine_tuned__confusion_matrix.png",
    "__baseline__metrics.json", "__fine_tuned__metrics.json",
]
for model_name in MODEL_NAMES:
    model_slug = model_name.replace("/", "__")
    existing_names = {path.name for path in RESULTS_DIR.glob(f"{model_slug}__*")}
    for suffix in required_suffixes:
        expected_name = model_slug + suffix
        assert expected_name in existing_names, f"Missing output: {expected_name}"
    for stage in ("baseline", "fine_tuned"):
        prediction_path, _ = prediction_paths(model_name, stage)
        df = pd.read_csv(prediction_path, dtype={"id": str})
        assert df["id"].tolist() == list(test_ds["id"]), f"Incomplete predictions: {prediction_path}"
        assert df["signature"].eq(expected_prediction_signature(stage)).all(), f"Stale predictions: {prediction_path}"

assert (RESULTS_DIR / "model_comparison_metrics.csv").exists()
assert (RESULTS_DIR / "macro_f1_comparison.png").exists()
assert (RESULTS_DIR / "run_manifest.json").exists()
assert set(train_ds["id"]).isdisjoint(validation_ds["id"])
assert set(train_ds["id"]).isdisjoint(test_ds["id"])
assert set(validation_ds["id"]).isdisjoint(test_ds["id"])
print("All expected outputs and resume invariants passed.")


,relative_path,size_bytes
0,Hate_Speech_Project_v1_7_Mistral_Only_TokenMas...,86290
1,cache/official_hatexplain_dataset/dataset_dict...,43
2,cache/official_hatexplain_dataset/test/data-00...,345800
3,cache/official_hatexplain_dataset/test/dataset...,446
4,cache/official_hatexplain_dataset/test/state.json,247
...,...,...
72,results/unsloth__mistral-7b-instruct-v0.3-bnb-...,68645
73,results/unsloth__mistral-7b-instruct-v0.3-bnb-...,451668
74,results/unsloth__mistral-7b-instruct-v0.3-bnb-...,445612
75,results/unsloth__mistral-7b-instruct-v0.3-bnb-...,357


All expected outputs and resume invariants passed.


## Cross-dataset generalization evaluation — Implicit Hate

This section does **not** retrain the model. It evaluates the existing HateXplain baseline and saved fine-tuned adapter on `tasksource/implicit-hate-stg1`.

Design:
- the original HateXplain prompt remains unchanged;
- model outputs are still `Hate`, `Normal`, or `Offensive`;
- for cross-dataset binary evaluation: `Hate -> HATE`, `Normal/Offensive -> NON-HATE`;
- external labels: `explicit_hate/implicit_hate -> HATE`, `not_hate -> NON-HATE`;
- the same deterministic relaxed parser is used for both model families:
  repeated identical valid labels are collapsed; conflicting valid labels remain invalid;
- the full external dataset is analyzed for descriptive statistics;
- inference uses a deterministic balanced sample with the size of the smallest source class, so all three external classes contribute equally;
- downloaded/processed data and partial predictions are cached;
- all upload-ready outputs are written to one `cross_dataset_results` directory.


In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

CROSS_DATASET_ID = "tasksource/implicit-hate-stg1"
CROSS_DATASET_SPLIT = "train"
CROSS_SAMPLE_SEED = 42

CROSS_RESULTS_DIR = PROJECT_ROOT / "cross_dataset_results"
CROSS_CM_DIR = CROSS_RESULTS_DIR / "confusion_matrices"
CROSS_CACHE_DIR = CACHE_DIR / "implicit_hate_stg1"
CROSS_FULL_CACHE_DIR = CROSS_CACHE_DIR / "full_train"
CROSS_SAMPLE_CACHE_DIR = CROSS_CACHE_DIR / f"balanced_eval_seed_{CROSS_SAMPLE_SEED}"

for directory in (CROSS_RESULTS_DIR, CROSS_CM_DIR, CROSS_CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def parse_label_relaxed(raw_output):
    """
    Deterministic parser used for cross-dataset evaluation in BOTH notebooks.

    - one valid label -> keep it
    - repeated instances of the same valid label -> collapse to one
    - two or more different valid labels -> invalid
    - no valid label -> invalid
    """
    if raw_output is None:
        return None

    matches = re.findall(
        r"\b(?:Hate|Normal|Offensive)\b",
        str(raw_output),
        flags=re.IGNORECASE,
    )
    if not matches:
        return None

    canonical = {
        "hate": "Hate",
        "normal": "Normal",
        "offensive": "Offensive",
    }
    normalized = [canonical[x.lower()] for x in matches]
    unique = set(normalized)

    if len(unique) != 1:
        return None
    return normalized[0]


def load_or_cache_implicit_hate():
    if CROSS_FULL_CACHE_DIR.exists():
        full_ds = load_from_disk(str(CROSS_FULL_CACHE_DIR))
        print("Loaded cached Implicit Hate dataset:", len(full_ds))
    else:
        full_ds = load_dataset(
            CROSS_DATASET_ID,
            split=CROSS_DATASET_SPLIT,
            token=hf_token,
        )
        tmp_dir = CROSS_FULL_CACHE_DIR.with_name(CROSS_FULL_CACHE_DIR.name + ".tmp")
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)
        full_ds.save_to_disk(str(tmp_dir))
        tmp_dir.replace(CROSS_FULL_CACHE_DIR)
        print("Downloaded and cached Implicit Hate dataset:", len(full_ds))

    required = {"post", "class"}
    missing = required - set(full_ds.column_names)
    if missing:
        raise RuntimeError(f"Unexpected Implicit Hate schema. Missing columns: {sorted(missing)}")

    return full_ds


def build_or_load_cross_eval_sample(full_ds):
    if CROSS_SAMPLE_CACHE_DIR.exists():
        sample_ds = load_from_disk(str(CROSS_SAMPLE_CACHE_DIR))
        print("Loaded cached balanced cross-dataset sample:", len(sample_ds))
        return sample_ds

    df = full_ds.to_pandas().reset_index(drop=True)
    df["source_row"] = np.arange(len(df))
    df["source_label"] = df["class"].astype(str)

    expected_labels = {"explicit_hate", "implicit_hate", "not_hate"}
    observed_labels = set(df["source_label"].unique())
    if observed_labels != expected_labels:
        raise RuntimeError(
            f"Unexpected source labels. Expected {sorted(expected_labels)}, "
            f"observed {sorted(observed_labels)}"
        )

    class_counts = df["source_label"].value_counts()
    per_class = int(class_counts.min())

    sampled_parts = []
    for source_label in sorted(expected_labels):
        part = (
            df.loc[df["source_label"] == source_label]
            .sample(n=per_class, random_state=CROSS_SAMPLE_SEED)
            .sort_values("source_row")
        )
        sampled_parts.append(part)

    sample_df = (
        pd.concat(sampled_parts, ignore_index=True)
        .sample(frac=1.0, random_state=CROSS_SAMPLE_SEED)
        .reset_index(drop=True)
    )

    sample_df["id"] = sample_df["source_row"].map(lambda x: f"implicit_hate_{int(x):06d}")
    sample_df["text"] = sample_df["post"].astype(str)
    sample_df["true_binary"] = sample_df["source_label"].map({
        "explicit_hate": 1,
        "implicit_hate": 1,
        "not_hate": 0,
    }).astype(int)
    sample_df["true_binary_label"] = sample_df["true_binary"].map({1: "HATE", 0: "NON-HATE"})

    sample_ds = Dataset.from_pandas(
        sample_df[
            ["id", "text", "source_label", "true_binary", "true_binary_label", "source_row"]
        ],
        preserve_index=False,
    )

    tmp_dir = CROSS_SAMPLE_CACHE_DIR.with_name(CROSS_SAMPLE_CACHE_DIR.name + ".tmp")
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    sample_ds.save_to_disk(str(tmp_dir))
    tmp_dir.replace(CROSS_SAMPLE_CACHE_DIR)

    print(
        f"Built balanced external sample: {len(sample_ds)} examples "
        f"({per_class} per source class)."
    )
    return sample_ds


implicit_hate_full = load_or_cache_implicit_hate()
cross_eval_ds = build_or_load_cross_eval_sample(implicit_hate_full)

# Save descriptive statistics from the FULL dataset and the actual evaluation sample.
full_stats_df = (
    implicit_hate_full.to_pandas()["class"]
    .astype(str)
    .value_counts()
    .rename_axis("source_label")
    .reset_index(name="count")
)
full_stats_df["subset"] = "full_dataset"
full_stats_df["proportion"] = full_stats_df["count"] / full_stats_df["count"].sum()

sample_stats_df = (
    pd.Series(cross_eval_ds["source_label"])
    .value_counts()
    .rename_axis("source_label")
    .reset_index(name="count")
)
sample_stats_df["subset"] = "balanced_evaluation_sample"
sample_stats_df["proportion"] = sample_stats_df["count"] / sample_stats_df["count"].sum()

dataset_statistics = pd.concat([full_stats_df, sample_stats_df], ignore_index=True)
dataset_statistics = dataset_statistics[
    ["subset", "source_label", "count", "proportion"]
]
atomic_write_csv(dataset_statistics, CROSS_RESULTS_DIR / "dataset_statistics.csv")

display(dataset_statistics)
print("Cross-dataset evaluation examples:", len(cross_eval_ds))


README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv:   0%|          | 0.00/2.19M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/21480 [00:00<?, ? examples/s]

Downloaded and cached Implicit Hate dataset: 21480


Saving the dataset (0/1 shards):   0%|          | 0/3267 [00:00<?, ? examples/s]

Built balanced external sample: 3267 examples (1089 per source class).


,subset,source_label,count,proportion
0,full_dataset,not_hate,13291,0.618762
1,full_dataset,implicit_hate,7100,0.330540
2,full_dataset,explicit_hate,1089,0.050698
3,balanced_evaluation_sample,not_hate,1089,0.333333
4,balanced_evaluation_sample,explicit_hate,1089,0.333333
5,balanced_evaluation_sample,implicit_hate,1089,0.333333


Cross-dataset evaluation examples: 3267


In [ ]:
CROSS_PARSER_VERSION = "relaxed_same_label_v1"
CROSS_LABEL_MAPPING = {
    "Hate": 1,
    "Normal": 0,
    "Offensive": 0,
}

CROSS_SIGNATURE_BASE = stable_hash({
    "dataset": CROSS_DATASET_ID,
    "split": CROSS_DATASET_SPLIT,
    "sample_seed": CROSS_SAMPLE_SEED,
    "sample_ids": list(cross_eval_ds["id"]),
    "output_mapping": CROSS_LABEL_MAPPING,
    "parser": CROSS_PARSER_VERSION,
    "prompt_parser_signature": PROMPT_PARSER_SIGNATURE,
})


def cross_signature(stage):
    if stage == "baseline":
        return stable_hash({
            "cross_base": CROSS_SIGNATURE_BASE,
            "stage": stage,
        })
    return stable_hash({
        "cross_base": CROSS_SIGNATURE_BASE,
        "stage": stage,
        "training_signature": TRAINING_SIGNATURE,
    })


def cross_paths(model_name, stage):
    slug = model_name.replace("/", "__")
    return {
        "predictions": CROSS_RESULTS_DIR / f"{slug}__{stage}__cross_predictions.csv",
        "invalid": CROSS_RESULTS_DIR / f"{slug}__{stage}__cross_invalid_outputs.csv",
        "errors": CROSS_RESULTS_DIR / f"{slug}__{stage}__cross_errors.csv",
        "metrics": CROSS_RESULTS_DIR / f"{slug}__{stage}__cross_metrics.json",
        "confusion": CROSS_CM_DIR / f"{slug}__{stage}__cross_confusion_matrix.png",
    }


def load_cross_prediction_cache(path, ds, model_name, stage):
    if not path.exists():
        return pd.DataFrame()

    try:
        df = pd.read_csv(path, dtype={"id": str})
    except Exception:
        return pd.DataFrame()

    required = {
        "id", "text", "source_label", "true_binary", "true_binary_label",
        "raw_output", "parsed_label", "prediction_binary", "prediction_binary_label",
        "is_valid", "model", "stage", "signature",
    }
    if not required.issubset(df.columns):
        return pd.DataFrame()

    expected_signature = cross_signature(stage)
    valid_ids = set(ds["id"])

    df = df.loc[
        df["model"].eq(model_name)
        & df["stage"].eq(stage)
        & df["signature"].eq(expected_signature)
        & df["id"].isin(valid_ids)
    ].drop_duplicates("id", keep="last")

    return df.copy()


@torch.inference_mode()
def generate_cross_predictions(model, tokenizer, ds, model_name, stage):
    paths = cross_paths(model_name, stage)
    predictions_df = load_cross_prediction_cache(
        paths["predictions"], ds, model_name, stage
    )

    completed_ids = set(predictions_df["id"]) if not predictions_df.empty else set()
    missing_indices = [
        i for i, item_id in enumerate(ds["id"])
        if item_id not in completed_ids
    ]

    if not missing_indices:
        print(
            f"{model_name} | {stage} | cross-dataset: "
            f"loaded {len(predictions_df)} cached predictions; inference skipped."
        )
        return predictions_df.set_index("id").loc[list(ds["id"])].reset_index()

    tokenizer.padding_side = "left"
    FastLanguageModel.for_inference(model)

    for offset in tqdm(
        range(0, len(missing_indices), INFERENCE_BATCH_SIZE),
        total=(len(missing_indices) + INFERENCE_BATCH_SIZE - 1) // INFERENCE_BATCH_SIZE,
        desc=f"{model_name} | {stage} | ImplicitHate",
    ):
        indices = missing_indices[offset:offset + INFERENCE_BATCH_SIZE]
        batch = ds.select(indices)

        # Keep the ORIGINAL HateXplain prompt unchanged.
        prompt_texts = [
            tokenizer.apply_chat_template(
                build_messages(text),
                tokenize=False,
                add_generation_prompt=True,
            )
            for text in batch["text"]
        ]

        encoded = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        )
        encoded = {k: v.to(model.device) for k, v in encoded.items()}

        generated = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        input_width = encoded["input_ids"].shape[1]
        records = []

        for i in range(len(batch)):
            raw_output = tokenizer.decode(
                generated[i, input_width:],
                skip_special_tokens=True,
            ).strip()

            parsed_label = parse_label_relaxed(raw_output)

            if parsed_label is None:
                prediction_binary = -1
                prediction_binary_label = "INVALID"
            else:
                prediction_binary = CROSS_LABEL_MAPPING[parsed_label]
                prediction_binary_label = "HATE" if prediction_binary == 1 else "NON-HATE"

            records.append({
                "id": batch["id"][i],
                "text": batch["text"][i],
                "source_label": batch["source_label"][i],
                "true_binary": int(batch["true_binary"][i]),
                "true_binary_label": batch["true_binary_label"][i],
                "raw_output": raw_output,
                "parsed_label": parsed_label,
                "prediction_binary": int(prediction_binary),
                "prediction_binary_label": prediction_binary_label,
                "is_valid": parsed_label is not None,
                "model": model_name,
                "stage": stage,
                "signature": cross_signature(stage),
            })

        predictions_df = pd.concat(
            [predictions_df, pd.DataFrame(records)],
            ignore_index=True,
        ).drop_duplicates("id", keep="last")

        existing_ids = set(predictions_df["id"])
        ordered_ids = [x for x in ds["id"] if x in existing_ids]
        ordered = (
            predictions_df
            .set_index("id")
            .reindex(ordered_ids)
            .reset_index()
        )

        atomic_write_csv(ordered, paths["predictions"])
        atomic_write_csv(
            ordered.loc[ordered["prediction_binary"].astype(int) == -1],
            paths["invalid"],
        )

        del encoded, generated
        torch.cuda.empty_cache()

    predictions_df = pd.read_csv(paths["predictions"], dtype={"id": str})

    if (
        len(predictions_df) != len(ds)
        or predictions_df["id"].tolist() != list(ds["id"])
    ):
        raise RuntimeError(
            f"Cross-dataset prediction cache incomplete or out of order: "
            f"{paths['predictions']}"
        )

    return predictions_df


def compute_cross_metrics(predictions_df):
    y_true = predictions_df["true_binary"].astype(int).to_numpy()
    y_pred = predictions_df["prediction_binary"].astype(int).to_numpy()

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        average="macro",
        zero_division=0,
    )

    hate_precision, hate_recall, hate_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[1],
        average=None,
        zero_division=0,
    )

    def source_success(source_label, expected_prediction):
        mask = predictions_df["source_label"].eq(source_label)
        if not mask.any():
            return float("nan")
        return float(
            predictions_df.loc[mask, "prediction_binary"]
            .astype(int)
            .eq(expected_prediction)
            .mean()
        )

    return {
        "model": predictions_df["model"].iloc[0],
        "stage": predictions_df["stage"].iloc[0],
        "dataset": CROSS_DATASET_ID,
        "evaluation_examples": int(len(predictions_df)),
        "binary_accuracy": float(accuracy_score(y_true, y_pred)),
        "binary_precision_macro": float(precision_macro),
        "binary_recall_macro": float(recall_macro),
        "binary_f1_macro": float(f1_macro),
        "hate_precision": float(hate_precision[0]),
        "hate_recall": float(hate_recall[0]),
        "hate_f1": float(hate_f1[0]),
        "explicit_hate_recall": source_success("explicit_hate", 1),
        "implicit_hate_recall": source_success("implicit_hate", 1),
        "not_hate_recall": source_success("not_hate", 0),
        "invalid_count": int((y_pred == -1).sum()),
        "invalid_rate": float((y_pred == -1).mean()),
        "signature": predictions_df["signature"].iloc[0],
    }


def save_cross_confusion_matrix(predictions_df, output_path):
    # Two true classes x three prediction outcomes, preserving INVALID explicitly.
    rows = ["NON-HATE", "HATE"]
    cols = ["NON-HATE", "HATE", "INVALID"]
    matrix = np.zeros((2, 3), dtype=int)

    for _, row in predictions_df.iterrows():
        true_idx = 1 if int(row["true_binary"]) == 1 else 0
        pred = int(row["prediction_binary"])
        pred_idx = {0: 0, 1: 1, -1: 2}[pred]
        matrix[true_idx, pred_idx] += 1

    fig, ax = plt.subplots(figsize=(6, 4))
    image = ax.imshow(matrix)
    ax.set_xticks(range(len(cols)), cols)
    ax.set_yticks(range(len(rows)), rows)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Cross-dataset confusion counts")

    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            ax.text(c, r, str(matrix[r, c]), ha="center", va="center")

    fig.colorbar(image, ax=ax)
    fig.tight_layout()

    tmp = output_path.with_suffix(".tmp.png")
    fig.savefig(tmp, dpi=200, bbox_inches="tight")
    plt.close(fig)
    tmp.replace(output_path)


def evaluate_and_save_cross_predictions(predictions_df):
    model_name = predictions_df["model"].iloc[0]
    stage = predictions_df["stage"].iloc[0]
    paths = cross_paths(model_name, stage)

    metrics = compute_cross_metrics(predictions_df)
    atomic_write_json(metrics, paths["metrics"])

    errors = predictions_df.loc[
        predictions_df["prediction_binary"].astype(int)
        != predictions_df["true_binary"].astype(int)
    ].copy()
    atomic_write_csv(errors, paths["errors"])

    invalid = predictions_df.loc[
        predictions_df["prediction_binary"].astype(int) == -1
    ].copy()
    atomic_write_csv(invalid, paths["invalid"])

    save_cross_confusion_matrix(predictions_df, paths["confusion"])
    return metrics


In [ ]:
def run_cross_dataset_experiment(model_name):
    """
    Cross-dataset evaluation only.
    No training is performed here.
    """
    model_slug = model_name.replace("/", "__")
    adapter_dir = RESULTS_DIR / model_slug / "best_adapter"
    adapter_marker = RESULTS_DIR / model_slug / "adapter_complete.json"

    if not adapter_is_complete(adapter_dir, adapter_marker, model_name):
        raise RuntimeError(
            "Completed fine-tuned adapter was not found. "
            "Run the existing HateXplain experiment first or restore its saved adapter. "
            "Cross-dataset evaluation will NOT retrain automatically."
        )

    metric_rows = []

    # Baseline
    model = tokenizer = None
    try:
        model, tokenizer = load_model(model_name)
        baseline_predictions = generate_cross_predictions(
            model, tokenizer, cross_eval_ds, model_name, "baseline"
        )
        metric_rows.append(
            evaluate_and_save_cross_predictions(baseline_predictions)
        )
    finally:
        model = tokenizer = None
        clear_gpu()

    # Fine-tuned adapter already trained on HateXplain
    model = tokenizer = None
    try:
        model, tokenizer = load_model(adapter_dir)
        fine_predictions = generate_cross_predictions(
            model, tokenizer, cross_eval_ds, model_name, "fine_tuned"
        )
        metric_rows.append(
            evaluate_and_save_cross_predictions(fine_predictions)
        )
    finally:
        model = tokenizer = None
        clear_gpu()

    # Merge into one shared summary file without deleting rows from the other notebook.
    summary_path = CROSS_RESULTS_DIR / "cross_dataset_summary.csv"
    new_df = pd.DataFrame(metric_rows)

    if summary_path.exists():
        old_df = pd.read_csv(summary_path)
        summary_df = pd.concat([old_df, new_df], ignore_index=True)
    else:
        summary_df = new_df

    summary_df = (
        summary_df
        .drop_duplicates(["model", "stage", "dataset"], keep="last")
        .sort_values(["model", "stage"])
        .reset_index(drop=True)
    )
    atomic_write_csv(summary_df, summary_path)

    manifest_path = CROSS_RESULTS_DIR / f"{model_slug}__cross_run_manifest.json"
    atomic_write_json({
        "dataset": CROSS_DATASET_ID,
        "dataset_split": CROSS_DATASET_SPLIT,
        "full_dataset_examples": len(implicit_hate_full),
        "balanced_evaluation_examples": len(cross_eval_ds),
        "sample_seed": CROSS_SAMPLE_SEED,
        "source_labels": ["explicit_hate", "implicit_hate", "not_hate"],
        "true_binary_mapping": {
            "explicit_hate": "HATE",
            "implicit_hate": "HATE",
            "not_hate": "NON-HATE",
        },
        "model_output_mapping": {
            "Hate": "HATE",
            "Normal": "NON-HATE",
            "Offensive": "NON-HATE",
        },
        "parser": CROSS_PARSER_VERSION,
        "prompt_changed": False,
        "training_performed_in_cross_dataset_section": False,
        "model": model_name,
        "training_signature": TRAINING_SIGNATURE,
        "cross_signature_base": CROSS_SIGNATURE_BASE,
    }, manifest_path)

    display(summary_df)
    print("All upload-ready cross-dataset outputs are in:", CROSS_RESULTS_DIR)
    return summary_df


cross_summary = run_cross_dataset_experiment(MODEL_NAMES[0])


==((====))==  Unsloth 2026.8.18: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

unsloth/mistral-7b-instruct-v0.3-bnb-4bit | baseline | ImplicitHate:   0%|          | 0/409 [00:00<?, ?it/s]

Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==((====))==  Unsloth 2026.8.18: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

Unsloth 2026.8.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


unsloth/mistral-7b-instruct-v0.3-bnb-4bit | fine_tuned | ImplicitHate:   0%|          | 0/409 [00:00<?, ?it/s]

Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=6) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,model,stage,dataset,evaluation_examples,binary_accuracy,binary_precision_macro,binary_recall_macro,binary_f1_macro,hate_precision,hate_recall,hate_f1,explicit_hate_recall,implicit_hate_recall,not_hate_recall,invalid_count,invalid_rate,signature
0,unsloth/mistral-7b-instruct-v0.3-bnb-4bit,baseline,tasksource/implicit-hate-stg1,3267,0.348944,0.483152,0.495868,0.295478,0.634921,0.055096,0.101394,0.072544,0.037649,0.936639,0,0.000000,e9de9dcb3a13dad6
1,unsloth/mistral-7b-instruct-v0.3-bnb-4bit,fine_tuned,tasksource/implicit-hate-stg1,3267,0.446893,0.709538,0.566116,0.472842,0.972163,0.208448,0.343289,0.342516,0.074380,0.923783,549,0.168044,cd6c1f423274b37a


All upload-ready cross-dataset outputs are in: /content/drive/MyDrive/Hate_Speech_Final_Project/cross_dataset_results
